# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip install -q duckdb huggingface_hub pandas numpy matplotlib

In [6]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add HF_TOKEN under Colab Secrets."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

DATA_PATH = (
    f"{HF_BASE}/fact_content_daily_performance/**/*.parquet"
)

print("Setup complete.")

Setup complete.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will prioritize pages that have demonstrated search visibility but show an opportunity for improvement in their current Google Search performance.

The baseline uses three observed signals:

* `gsc_impressions` — search visibility/demand
* `gsc_avg_position` — current average search position
* `gsc_clicks / gsc_impressions` — observed CTR

Pages with higher impressions, non-top search positions, and relatively low CTR receive higher scores.

This is a transparent decision-support rule for prioritizing pages for review. It is not a prediction of Google's ranking algorithm or proof that a page needs a refresh.

### Reason codes

* `visible_low_ctr` — high search visibility with relatively low CTR
* `visible_position_opportunity` — visible page with an average position outside the strongest top positions
* `high_visibility_review` — high visibility but does not meet the stronger reason-code conditions
* `insufficient_search_data` — insufficient GSC data to score reliably


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Data availability check

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# Check that all required columns exist
available_columns = con.sql(f"""
SELECT *
FROM read_parquet('{DATA_PATH}')
LIMIT 1
""").df().columns.tolist()

missing_columns = [
    col for col in required_columns
    if col not in available_columns
]

print("Required columns:", required_columns)
print("Missing columns:", missing_columns)

assert not missing_columns, f"Missing columns: {missing_columns}"

print("All required columns are available.")

Required columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position']
Missing columns: []
All required columns are available.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

##Baseline Score

The score combines three current-window signals:

1. Search visibility from `gsc_impressions`.
2. Position opportunity from `gsc_avg_position`.
3. Low CTR relative to the observed dataset.

Higher scores indicate pages that are more suitable for review under this baseline rule.

The score is a ranking heuristic, not a supervised model or causal estimate.


In [9]:
# Section 2 — Build ranked queue (optimized)

import os

os.makedirs("work/outputs", exist_ok=True)

queue_path = "work/outputs/baseline_action_score.csv"

query = f"""
COPY (
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY baseline_score DESC,
                     gsc_impressions DESC
        ) AS rank,

        content_hash_id,
        client_hash_id,
        report_date,

        ROUND(baseline_score, 4) AS baseline_score,

        CASE
            WHEN gsc_impressions >= 500
                 AND ctr < 0.01
                 AND gsc_avg_position <= 20
                THEN 'visible_low_ctr'

            WHEN gsc_impressions >= 500
                 AND gsc_avg_position > 10
                 AND gsc_avg_position <= 20
                THEN 'visible_position_opportunity'

            ELSE 'high_visibility_review'
        END AS reason_code,

        'review_search_performance' AS action,

        gsc_impressions,
        gsc_clicks,
        ROUND(ctr * 100, 4) AS ctr_percent,
        ROUND(gsc_avg_position, 2) AS gsc_avg_position

    FROM (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,

            -- Simple CTR
            gsc_clicks / NULLIF(gsc_impressions, 0) AS ctr,

            -- Simple transparent baseline score
            100 * (
                0.40 * LEAST(LN(1 + gsc_impressions) / LN(501), 1.0)
                +
                0.35 * CASE
                    WHEN gsc_avg_position <= 10 THEN 0.0
                    WHEN gsc_avg_position >= 20 THEN 1.0
                    ELSE (gsc_avg_position - 10) / 10.0
                END
                +
                0.25 * CASE
                    WHEN gsc_clicks / NULLIF(gsc_impressions, 0) < 0.01
                    THEN 1.0
                    ELSE 0.0
                END
            ) AS baseline_score

        FROM read_parquet('{DATA_PATH}')

        WHERE gsc_impressions >= 100
          AND gsc_clicks IS NOT NULL
          AND gsc_avg_position IS NOT NULL
    )

    ORDER BY baseline_score DESC,
             gsc_impressions DESC

    LIMIT 20
)
TO '{queue_path}'
WITH (
    FORMAT CSV,
    HEADER TRUE
);
"""

con.execute(query)

print("Top-20 queue written to:", queue_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top-20 queue written to: work/outputs/baseline_action_score.csv


In [10]:
queue = pd.read_csv(queue_path)

print("Rows:", len(queue))
display(queue)

Rows: 20


,rank,content_hash_id,client_hash_id,report_date,baseline_score,reason_code,action,gsc_impressions,gsc_clicks,ctr_percent,gsc_avg_position
0,1,content_c60628276389acbb,client_23a62021009f63c4,2026-05-19,100.0,high_visibility_review,review_search_performance,19846,1,0.0050,89.73
1,2,content_c60628276389acbb,client_23a62021009f63c4,2026-05-20,100.0,high_visibility_review,review_search_performance,17060,0,0.0000,90.55
2,3,content_c60628276389acbb,client_23a62021009f63c4,2026-06-01,100.0,high_visibility_review,review_search_performance,14920,0,0.0000,85.76
3,4,content_c60628276389acbb,client_23a62021009f63c4,2026-05-21,100.0,high_visibility_review,review_search_performance,14798,0,0.0000,89.00
4,5,content_c60628276389acbb,client_23a62021009f63c4,2026-05-18,100.0,high_visibility_review,review_search_performance,14451,1,0.0069,87.76
5,6,content_a9322b74ca7cb1bb,client_23a62021009f63c4,2026-06-01,100.0,high_visibility_review,review_search_performance,14148,0,0.0000,90.63
6,7,content_a9322b74ca7cb1bb,client_23a62021009f63c4,2026-05-30,100.0,high_visibility_review,review_search_performance,14017,0,0.0000,86.58
7,8,content_c60628276389acbb,client_23a62021009f63c4,2026-05-30,100.0,high_visibility_review,review_search_performance,14015,2,0.0143,86.15
8,9,content_c60628276389acbb,client_23a62021009f63c4,2026-05-31,100.0,high_visibility_review,review_search_performance,13692,0,0.0000,84.26
9,10,content_a9322b74ca7cb1bb,client_23a62021009f63c4,2026-05-31,100.0,high_visibility_review,review_search_performance,13614,0,0.0000,84.27


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The following table contains the 20 highest-ranked pages under the baseline rule.

The confidence note describes why the rule considers each page worth reviewing based on observed search data. It should not be interpreted as certainty that the page requires a content refresh.

Possible reasons a high-ranked page could still be a weak pick include changes in search demand, a legitimate low CTR caused by the SERP layout, brand/non-brand query mix, or incomplete GSC measurements.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 review

top20 = pd.read_csv(
    queue_path,
    nrows=20
)

def confidence_note(row):
    if row["reason_code"] == "visible_low_ctr":
        return (
            "High visibility with relatively low observed CTR; "
            "worth reviewing title/snippet/search intent alignment."
        )

    if row["reason_code"] == "visible_position_opportunity":
        return (
            "Visible page with average position in the opportunity range; "
            "worth reviewing content and search-result performance."
        )

    return (
        "High search visibility makes this page useful for review, "
        "but the baseline alone does not prove a refresh is needed."
    )


def wrong_if(row):
    if row["reason_code"] == "visible_low_ctr":
        return (
            "Could be wrong if the low CTR is explained by SERP features, "
            "query mix, or intentional search-result behaviour."
        )

    if row["reason_code"] == "visible_position_opportunity":
        return (
            "Could be wrong if the observed position reflects a stable "
            "competitive/query mix rather than a content opportunity."
        )

    return (
        "Could be wrong if high impressions reflect demand without "
        "a realistic opportunity for improvement."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "baseline_score",
    "gsc_impressions",
    "gsc_clicks",
    "ctr_percent",
    "gsc_avg_position",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,action,reason_code,baseline_score,gsc_impressions,gsc_clicks,ctr_percent,gsc_avg_position,confidence_note,what_would_make_it_wrong
0,1,review_search_performance,high_visibility_review,100.0,19846,1,0.0050,89.73,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
1,2,review_search_performance,high_visibility_review,100.0,17060,0,0.0000,90.55,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
2,3,review_search_performance,high_visibility_review,100.0,14920,0,0.0000,85.76,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
3,4,review_search_performance,high_visibility_review,100.0,14798,0,0.0000,89.00,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
4,5,review_search_performance,high_visibility_review,100.0,14451,1,0.0069,87.76,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
5,6,review_search_performance,high_visibility_review,100.0,14148,0,0.0000,90.63,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
6,7,review_search_performance,high_visibility_review,100.0,14017,0,0.0000,86.58,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
7,8,review_search_performance,high_visibility_review,100.0,14015,2,0.0143,86.15,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
8,9,review_search_performance,high_visibility_review,100.0,13692,0,0.0000,84.26,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...
9,10,review_search_performance,high_visibility_review,100.0,13614,0,0.0000,84.27,High search visibility makes this page useful ...,Could be wrong if high impressions reflect dem...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some high-ranked pages may be weak picks even when their observed search metrics look favourable.

A page can have high impressions and low CTR because of SERP features, query intent, brand/non-brand differences, or the way the search result is displayed. A position between 10 and 20 can also reflect competition rather than a content-quality problem.

Therefore, the baseline should be used as a review queue rather than an automatic refresh decision.

### Leakage check

The baseline uses only current observed fields from the warehouse:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`

CTR is calculated directly from current GSC impressions and clicks.

No future-window performance, model prediction, product decision flag, or target-derived field is used.

The score therefore represents a current-data prioritization heuristic rather than a future prediction.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

# Show the lowest-confidence / potentially weak cases among the Top-20.
# These are intentionally inspected rather than hidden.

weak_picks = top20[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_percent",
        "gsc_avg_position"
    ]
].copy()

print("Potential weak picks for manual review:")
display(weak_picks)

# -----------------------------
# Leakage / feature audit
# -----------------------------

used_features = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr"  # derived from clicks / impressions
}

forbidden_or_unavailable_features = {
    "future_impressions",
    "future_clicks",
    "future_ctr",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
    "refresh_tier",
    "model_prediction"
}

print("\nUsed baseline features:")
for feature in sorted(used_features):
    print("-", feature)

print("\nLeakage check:")
print("Future-window fields used: NO")
print("Product decision flags used: NO")
print("Model predictions used: NO")
print("Target-derived fields used: NO")

assert not (
    used_features & forbidden_or_unavailable_features
), "Potential leakage detected."

print("\nLeakage check passed.")

Potential weak picks for manual review:


,rank,content_hash_id,baseline_score,reason_code,gsc_impressions,gsc_clicks,ctr_percent,gsc_avg_position
0,1,content_c60628276389acbb,100.0,high_visibility_review,19846,1,0.0050,89.73
1,2,content_c60628276389acbb,100.0,high_visibility_review,17060,0,0.0000,90.55
2,3,content_c60628276389acbb,100.0,high_visibility_review,14920,0,0.0000,85.76
3,4,content_c60628276389acbb,100.0,high_visibility_review,14798,0,0.0000,89.00
4,5,content_c60628276389acbb,100.0,high_visibility_review,14451,1,0.0069,87.76
5,6,content_a9322b74ca7cb1bb,100.0,high_visibility_review,14148,0,0.0000,90.63
6,7,content_a9322b74ca7cb1bb,100.0,high_visibility_review,14017,0,0.0000,86.58
7,8,content_c60628276389acbb,100.0,high_visibility_review,14015,2,0.0143,86.15
8,9,content_c60628276389acbb,100.0,high_visibility_review,13692,0,0.0000,84.26
9,10,content_a9322b74ca7cb1bb,100.0,high_visibility_review,13614,0,0.0000,84.27



Used baseline features:
- ctr
- gsc_avg_position
- gsc_clicks
- gsc_impressions

Leakage check:
Future-window fields used: NO
Product decision flags used: NO
Model predictions used: NO
Target-derived fields used: NO

Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.